# Model evaluation and comparison

**Data sources used in this notebook:**
- `sudan_results.csv` - current set of runs (k=0.5/1.0 only, post threshold-fix, active window extended to include 2025)
- `sudan_results_historical.csv` - earlier runs including k=0.25, used only for the k-selection comparison in Section 1
- `ethiopia_results.csv` — Ethiopia/Tigray replication runs

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from utils.constants import ONSET_END_DATE, ONSET_START_DATE
from utils.data_prep import get_clean_combined_data

sudan = pd.read_csv("evaluation/sudan_results_combined.csv")
# sudan = pd.read_csv("evaluation/sudan_results.csv")
# sudan = pd.read_csv("evaluation/sudan_results_historical.csv") # Before the collapse
ethiopia = pd.read_csv("evaluation/ethiopia_results.csv")

print(f"Sudan: {len(sudan)} runs")
print(f"Ethiopia: {len(ethiopia)} runs")

Sudan: 1013 runs
Ethiopia: 32 runs


# 1. Choosing the k escalation threshold
## 1.1 Rejecting k=0.25

`k` controls the escalation threshold — a lower k means a looser threshold (escalation is easier to trigger).

**Issue**

Raw AUPR favours the lowest `k`, but that is misleading for conflict prediction. The model was catching 100% of true positives by default rather than through genuine skill. Domain knowledge also matters here - in conflict forecasting, you want a model that's sensitive to real escalations without flagging every small, ordinary shift in conflict as significant.

Three values of `k` (0.25, 0.5, 1.0) were tested in the initial sweep, measuring both the true onset-window prevalence of escalation and the rate at which the F1-optimal threshold collapsed to predicting positive for nearly every region-month.

`k`=0.25 had the highest prevalence (42.1%) and the highest collapse rate (75.0%), making it a poor definition of true escalation — closer to a coin flip than a meaningful departure from a region's baseline. `k`=0.5 was somewhat better (37.0% prevalence, 57.2% collapse) but still substantially collapsed. This motivated dropping `k`=0.25 from further consideration, and investigating the collapse behaviour more closely — covered in Section 1.2.

In [35]:
prevalence_records = []
for k_test in [0.25, 0.5, 1]:
    model_data, predictor_cols = get_clean_combined_data(
        data_sources=[], k=k_test, event_col="sub_event_type", conflict_only_embeddings=True,
    )
    onset_slice = model_data[
        (model_data["year_month"] >= pd.Period(ONSET_START_DATE, freq="M"))
        & (model_data["year_month"] <= pd.Period(ONSET_END_DATE, freq="M"))
    ]
    n_total = len(onset_slice)
    n_escalations = int(onset_slice["target_escalation"].sum())
    prevalence_records.append({
        "k": k_test,
        "onset_prevalence_pct": round(n_escalations / n_total * 100, 1),
        "n_escalations": n_escalations,
        "n_onset_rows": n_total,
    })
prevalence_df_early = pd.DataFrame(prevalence_records).set_index("k")

sudan_pre_fix = sudan[sudan["threshold_fix_applied"] == False]

collapse_rate_pre_fix = sudan_pre_fix.groupby("k")["onset_recall_class1"].apply(lambda x: (x == 1.0).mean())

k_summary_1_1 = pd.DataFrame({
    "mean_onset_aupr": sudan_pre_fix.groupby("k")["onset_aupr"].mean(),
    "max_onset_aupr": sudan_pre_fix.groupby("k")["onset_aupr"].max(),
    "collapse_rate": collapse_rate_pre_fix,
})
k_summary_1_1 = k_summary_1_1.join(prevalence_df_early[["onset_prevalence_pct"]])
k_summary_1_1.round(3)

INFO:AcledClient:Access token correctly retrieved.
INFO:AcledClient:Reading data from data/acled/acled_sudan.csv.
INFO:Events processing:Data grouped by sub_event_type
INFO:Events processing:Escalation target set at 0.25 standard deviations above the mean.
INFO:Data preparation:ACLED data processed.
INFO:AcledClient:Access token correctly retrieved.
INFO:AcledClient:Reading data from data/acled/acled_sudan.csv.
INFO:Events processing:Data grouped by sub_event_type
INFO:Events processing:Escalation target set at 0.5 standard deviations above the mean.
INFO:Data preparation:ACLED data processed.
INFO:AcledClient:Access token correctly retrieved.
INFO:AcledClient:Reading data from data/acled/acled_sudan.csv.
INFO:Events processing:Data grouped by sub_event_type
INFO:Events processing:Escalation target set at 1 standard deviations above the mean.
INFO:Data preparation:ACLED data processed.


,mean_onset_aupr,max_onset_aupr,collapse_rate,onset_prevalence_pct
k,,,,
0.25,0.440,0.526,0.750,42.1
0.50,0.369,0.458,0.662,37.0
1.00,0.341,0.421,0.338,30.6


In [37]:
ks = k_summary_1_1.index.astype(str)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Positive-class prevalence by k", "Threshold-collapse rate (%)"))

fig.add_trace(
    go.Bar(x=ks, y=k_summary_1_1["onset_prevalence_pct"], marker_color="#898781", name="Prevalence"),
    row=1, col=1,
)
fig.add_trace(
    go.Bar(x=ks, y=k_summary_1_1["collapse_rate"] * 100, marker_color="#c0392b", name="Collapse rate"),
    row=1, col=2,
)

fig.update_xaxes(title_text="k", row=1, col=1)
fig.update_xaxes(title_text="k", row=1, col=2)
fig.update_yaxes(title_text="% of onset rows that were escalations", row=1, col=1)
fig.update_yaxes(title_text="% of runs with recall=1.0", row=1, col=2)

fig.update_layout(
    showlegend=False, width=900, height=450,
    plot_bgcolor="white", paper_bgcolor="white",
    margin=dict(t=120),
    title={
        "text": "k=0.25 had the highest prevalence and the highest collapse rate,<br>before the threshold-collapse fix was applied",
        "y": 0.9, "yanchor": "top",
    },
)
fig.show()

## 1.2 The threshold-collapse bug and fix

**Bug**

In the initial set of runs, `optimal_threshold = thresholds[np.argmax(f1_scores)]` picked the lowest tied threshold whenever F1 plateaued, producing "predict everything positive" models. This was discovered because many runs had recall=1 and precision equal to the actual proportion of escalations in the data.

**Fix**

Instead, the model picks the highest threshold among those tied for the best F1 score, making it more conservative and less likely to default to predicting everything positive.

```python
max_f1 = f1_scores.max()
tied_indices = np.flatnonzero(f1_scores == max_f1)
optimal_threshold = thresholds[tied_indices[-1]]
```

**Effect of the fix**

Holding `k` constant (0.5 and 1.0 - as 0.25 was dropped), the collapse rate fell from 50.0% to 40.6%. This is an improvement, but it did not eliminate the issue at the looser threshold. Therefore further work was required to investigate raising `k`, covered in Section 1.3.

In [38]:
pre_threshold_fix = sudan[sudan["threshold_fix_applied"] == False]
threshold_fix = sudan[sudan["threshold_fix_applied"] == True]

pre_fix_shared = pre_threshold_fix[pre_threshold_fix["k"].isin([0.5, 1.0])]
post_fix_shared = threshold_fix[threshold_fix["k"].isin([0.5, 1.0])]

pre_fix_collapse_rate = (pre_fix_shared["onset_recall_class1"] == 1.0).mean()
post_fix_collapse_rate = (post_fix_shared["onset_recall_class1"] == 1.0).mean()

print(f"Pre-fix collapse rate (k=0.5/1.0 only): {pre_fix_collapse_rate:.1%}")
print(f"Post-fix collapse rate (k=0.5/1.0 only): {post_fix_collapse_rate:.1%}")

Pre-fix collapse rate (k=0.5/1.0 only): 50.0%
Post-fix collapse rate (k=0.5/1.0 only): 40.6%
